# BIST Hisseleri ve TCMB Döviz Kuru Verileriyle Risk-Getiri Analizi

## 1. Proje Tanımı

Bu notebook, seçilmiş BIST hisseleri ve TCMB USD/TRY kuru verileri üzerinde yapılacak veri bilimi dönem projesinin ana çalışma dosyasıdır.

Analiz günlük getiri, volatilite/risk, korelasyon ve basit regresyon modeline odaklanacaktır.

Notebook adım adım geliştirilecektir. Bu aşamada notebook iskeletine ham veri yükleme akışı eklenmiştir.

Proje eğitim amaçlıdır ve yatırım tavsiyesi değildir.

## 2. Veri Kaynakları ve Kullanılan Finansal Varlıklar

Projede seçilen BIST hisseleri ile TCMB USD/TRY döviz kuru verilerinin kullanılması planlanmaktadır.

Planlanan BIST hisseleri: THYAO.IS, ASELS.IS, GARAN.IS, SISE.IS ve KCHOL.IS.

Döviz kuru değişkeni olarak TCMB USD/TRY kuru kullanılacaktır.

## 3. Kütüphanelerin Yüklenmesi

Bu bölümde proje boyunca kullanılacak temel veri analizi ve görselleştirme kütüphaneleri yüklenecektir.

Bu aşamada temel kütüphanelere ek olarak BIST verisini indirmek için `yfinance` ve proje yollarını yönetmek için `pathlib` eklenmiştir. Modelleme kütüphaneleri ileriki adımlarda eklenecektir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import yfinance as yf
from pathlib import Path

## 4. Veri Yükleme Planı

Bu bölümde yalnızca ham veri yükleme akışı hazırlanacaktır.

BIST hisse fiyatları Yahoo Finance üzerinden `yfinance` kütüphanesi ile indirilecektir. TCMB USD/TRY kuru verisi ise TCMB kaynağından alınmış yerel bir CSV dosyası olarak beklenmektedir.

Bu adımda veri temizleme, veri birleştirme, günlük getiri hesaplama, grafik oluşturma veya modelleme yapılmayacaktır. Bu işlemler ileriki adımlarda ele alınacaktır.

In [ ]:
current_path = Path.cwd().resolve()

if current_path.name == "notebooks":
    PROJECT_ROOT = current_path.parent
else:
    PROJECT_ROOT = current_path

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

START_DATE = "2022-01-01"
END_DATE = "2026-06-30"
BIST_TICKERS = ["THYAO.IS", "ASELS.IS", "GARAN.IS", "SISE.IS", "KCHOL.IS"]

print(f"Proje ana klasörü: {PROJECT_ROOT}")
print(f"Ham veri klasörü: {RAW_DATA_DIR}")

### BIST Hisse Verilerinin Yüklenmesi

Seçilen BIST hisselerinin fiyat verileri Yahoo Finance üzerinden indirilecektir. Bu aşamada yalnızca kapanış fiyatları seçilecek ve ham veri olarak `data/raw/bist_close_prices_raw.csv` dosyasına kaydedilecektir.

Bu bölümde günlük getiri hesaplaması veya görselleştirme yapılmayacaktır.

In [ ]:
yfinance_end_date = (pd.to_datetime(END_DATE) + pd.Timedelta(days=1)).strftime("%Y-%m-%d")

bist_raw = yf.download(
    tickers=BIST_TICKERS,
    start=START_DATE,
    end=yfinance_end_date,
    auto_adjust=True,
    progress=False
)

if isinstance(bist_raw.columns, pd.MultiIndex):
    if "Close" in bist_raw.columns.get_level_values(0):
        bist_close = bist_raw["Close"].copy()
    elif "Close" in bist_raw.columns.get_level_values(1):
        bist_close = bist_raw.xs("Close", axis=1, level=1).copy()
    else:
        raise ValueError("İndirilen BIST verisinde kapanış fiyatı sütunu bulunamadı.")
else:
    bist_close = bist_raw[["Close"]].copy()

bist_close.index.name = "Date"
bist_close_path = RAW_DATA_DIR / "bist_close_prices_raw.csv"
bist_close.to_csv(bist_close_path)

print(f"BIST kapanış fiyatları kaydedildi: {bist_close_path}")
print(f"BIST veri boyutu: {bist_close.shape}")
bist_close.head()

### TCMB USD/TRY Verisinin Yüklenmesi

Döviz kuru verisinin TCMB kaynağından alınmış yerel bir CSV dosyası olarak `data/raw/tcmb_usdtry.csv` konumunda bulunması beklenmektedir.

CSV dosyasında tarih bilgisini içeren bir sütun ve USD/TRY kur değerini içeren bir değer sütunu olması beklenmektedir.

Bu adımda yalnızca dosyanın yüklenip yüklenemediği kontrol edilecektir. Sütun adlarının standartlaştırılması, eksik değer kontrolleri, BIST verisiyle birleştirme ve günlük getiri hesaplama işlemleri sonraki adımlarda yapılacaktır.

In [ ]:
tcmb_path = RAW_DATA_DIR / "tcmb_usdtry.csv"

if tcmb_path.exists():
    tcmb_raw = pd.read_csv(tcmb_path)
    print(f"TCMB USD/TRY ham verisi yüklendi: {tcmb_path}")
    print(f"TCMB veri boyutu: {tcmb_raw.shape}")
else:
    print("TCMB USD/TRY CSV dosyası bulunamadı.")
    print(f"Lütfen dosyayı şu konuma yerleştirin: {tcmb_path}")
    tcmb_raw = pd.DataFrame()

tcmb_raw.head()

## 5. Veri Temizleme Planı

Bu bölümde eksik değerler, tarih alanları, veri tipleri ve olası uyumsuzluklar kontrol edilecektir.

Veri temizleme adımı, veri yükleme aşamasından sonra uygulanacaktır.

## 6. Günlük Getiri Hesaplama Planı

Bu bölümde hisse fiyatları ve döviz kuru için günlük değişim/getiri oranlarının hesaplanması planlanmaktadır.

Günlük getiriler, risk ve ilişki analizlerinin temel değişkenleri olacaktır.

## 7. Keşifsel Veri Analizi Planı

Bu bölümde temel istatistikler, dağılımlar, volatilite karşılaştırmaları ve değişkenler arasındaki ilişkiler incelenecektir.

EDA aşaması, araştırma sorularını yanıtlamaya yardımcı olacak şekilde sade tutulacaktır.

## 8. Görselleştirme Planı

Bu bölümde zaman serisi grafikleri, dağılım grafikleri, korelasyon ısı haritası ve risk-getiri karşılaştırmaları gibi görselleştirmeler planlanmaktadır.

Bu aşamada grafik oluşturulmamıştır.

## 9. Araştırma Soruları

Bu projede aşağıdaki araştırma sorularının yanıtlanması planlanmaktadır:

1. Seçilen BIST hisselerinin günlük getiri dağılımları nasıldır?
2. Hangi hisse daha yüksek risk/volatilite göstermektedir?
3. TCMB USD/TRY kuru ile BIST hisselerinin günlük getirileri arasında ilişki var mıdır?
4. Seçilen BIST hisseleri birbirleriyle benzer hareket ediyor mu?

## 10. Basit Modelleme Planı

Bu bölümde ileriki aşamada basit ve açıklanabilir bir regresyon modeli kurulması planlanmaktadır.

Modelleme adımı henüz uygulanmamıştır. Bu aşamada scikit-learn import edilmemiştir.

## 11. Bulguların Yorumlanması

Analiz tamamlandıktan sonra elde edilen bulgular finans ve ekonomi teması bağlamında yorumlanacaktır.

Yorumlar, model sonuçlarının ve görselleştirmelerin sınırlılıkları dikkate alınarak yapılacaktır.

## 12. Sınırlamalar

Finansal günlük getiriler gürültülü olabilir ve kısa vadeli tahminlerde yüksek doğruluk beklenmemelidir.

Bu proje eğitim amaçlıdır ve yatırım tavsiyesi değildir. Kullanılacak model basit ve açıklanabilir tutulacaktır.

## 13. Sonuç

Bu notebook iskeleti, projenin sonraki veri yükleme, temizleme, analiz, görselleştirme ve modelleme adımları için temel yapıyı oluşturur.

İleriki adımlarda her bölüm sırayla geliştirilecektir.